In [57]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


---
---
# 🔬 Rentals study
---

In [58]:
# Import data
path_to_data =  f"../data/raw/get_around_delay_analysis.xlsx"
df_rentals = pd.read_excel(path_to_data)
df_rentals.head(30)

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN
5,511626,398802,mobile,ended,-203.0,NaN,NaN
6,511639,370585,connect,ended,-15.0,563782.0,570.0
7,512303,371242,mobile,ended,-44.0,NaN,NaN
8,512475,322502,mobile,canceled,NaN,NaN,NaN
9,513434,256528,connect,ended,23.0,NaN,NaN


Column type modification:
- Columns ```delay_at_checkout_in_minutes``` and ```time_delta_with_previous_rental_in_minutes``` should be of integer type (= time in plain minutes) 
- Column ```previous_ended_rental_id``` should be an integer (= rental_id)  

In [59]:
# Convert float columns to int
df_rentals['delay_at_checkout_in_minutes'] = df_rentals['delay_at_checkout_in_minutes'].astype('Int64')
df_rentals['previous_ended_rental_id'] = df_rentals['previous_ended_rental_id'].astype('Int64')
df_rentals['time_delta_with_previous_rental_in_minutes'] = df_rentals['time_delta_with_previous_rental_in_minutes'].astype('Int64')

Column renaming:
- ```delay_at_checkout_in_minutes``` -> ```delay_at_checkout```
- ```time_delta_with_previous_rental_in_minutes``` -> ```delta_with_previous```

In [4]:
df_rentals.rename(columns={
    "delay_at_checkout_in_minutes":"delay_at_checkout",
    "time_delta_with_previous_rental_in_minutes":"delta_with_previous"
    }, inplace=True)

In [5]:
df_rentals.describe(include='all')

,rental_id,car_id,checkin_type,state,delay_at_checkout,previous_ended_rental_id,delta_with_previous
count,21310.000000,21310.000000,21310,21310,16346.0,1841.0,1841.0
unique,NaN,NaN,2,2,<NA>,<NA>,<NA>
top,NaN,NaN,mobile,ended,<NA>,<NA>,<NA>
freq,NaN,NaN,17003,18045,<NA>,<NA>,<NA>
mean,549712.880338,350030.603426,NaN,NaN,59.701517,550127.411733,279.28843
std,13863.446964,58206.249765,NaN,NaN,1002.561635,13184.023111,254.594486
min,504806.000000,159250.000000,NaN,NaN,-22433.0,505628.0,0.0
25%,540613.250000,317639.000000,NaN,NaN,-36.0,540896.0,60.0
50%,550350.000000,368717.000000,NaN,NaN,9.0,550567.0,180.0
75%,560468.500000,394928.000000,NaN,NaN,67.0,560823.0,540.0


### Columns informations:

- ```car_id```: no link between the pricing dataset
- ```checkin_type```: type of medium used for rental: mobile/connect
- ```state``` : ended /canceled 
- ```delay_at_checkout```: time in min: difference between the real checkout time and the scheduled checkout. Negative means checkout on time
- ```delta_with_previous```: scheduled delta in min between 2 rentals: max 720 min (= 12 hours) between 2 rentals. If delta > 720, then the value might be missing (new car or car rented first time in the day)
- ```previous_ended_rental_id```: id of the previous rental this day: some cars are rented several times a day. For a new car, or for the first rental this day, the value might be missing

---
### Missing values:

In [6]:
# Missing values
print("Percentage of missing values: ")
display(100 * df_rentals.isnull().sum() / df_rentals.shape[0])

Percentage of missing values: 


rental_id                    0.000000
car_id                       0.000000
checkin_type                 0.000000
state                        0.000000
delay_at_checkout           23.294228
previous_ended_rental_id    91.360863
delta_with_previous         91.360863
dtype: float64

* No missing values in:
rental_id, car_id, checkin_type, state  
Each rental correspond to one car, that has one out of two checkin types, in one out of two states of the rental process

* Missing values in:  
delay_at_checkout, previous_ended_rental_id, delta_with_previous

In [7]:
# Get the max in delta_with_previous
print(f"Maximum delay between 2 rentals: {df_rentals["delta_with_previous"].max()} min")

Maximum delay between 2 rentals: 720 min


Drop columns ?  
criterions:  
all unique, more than 70% of missing, cardinality too important (too many distinct nominal values), colinearity  
-> No col drop

---
### 🔎 ```state``` column analysis

In [8]:
# Display the repartition of rentals between canceled vs. ended
fig = px.pie(
    df_rentals,
    names="state",
    title="Rentals repartition by state",
    width=600,
    height=250
    )
# Margin reduction
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )

- There are 2 different values in the column: "ended"/"canceled".
- 15% of rentals are canceled.

---
### 🔎 ```delay_at_checkout``` analysis

In [9]:
# Visualize by 12 hours bloc: delay_at_checkout <= 0 mean the car is checked out on time

# Discard NaN in "delay_at_checkout" column
df_delay_at_checkout = df_rentals.loc[df_rentals["delay_at_checkout"].notna(), ["delay_at_checkout", "state"]].copy()

# Select delay blocks of 720 min 
DELAY_SPLIT = 720

# Split delay_at_checkout into blocks
df_delay_at_checkout["delay_split"] = df_delay_at_checkout["delay_at_checkout"] // DELAY_SPLIT

fig = px.histogram(
    df_delay_at_checkout,
    x="delay_split",
    color="state",
    barmode="overlay",
    title="Distribution of delay_at_checkout (bins of 12h)",
    labels={"delay_split": "Delay interval (bloc of 720 min)"},
    width=800,
    height=250
)
# Margin reduction
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

- The majority of delays range from -1440 to 1440 ([-24h ; +24h])

##### Outliers in ```delay_at_checkout```

For canceled rentals, there should not be any checkout (```delay_at_checkout``` value is not missing)

In [10]:
# Are there "canceled" rentals for which delay_at_checkout is not missing ?
df_canceled = df_rentals[df_rentals['state'] == "canceled"].copy()
count_delay_not_missing = df_canceled["delay_at_checkout"].notna().sum()
print(f"count_delay_not_missing: {count_delay_not_missing}")
del(df_canceled)

count_delay_not_missing: 1


-> Delete this line.

In [11]:
# Delete the line
rentals_count = df_rentals.shape[0]
# mask the line that are not "canceled" with a checkout time
mask = ~((df_rentals["state"] == "canceled") & (df_rentals["delay_at_checkout"].notna()))
df_rentals = df_rentals.loc[mask, :].copy()
print(f"Canceled rentals before deletion: {rentals_count} - Canceled rentals after deletion: {df_rentals.shape[0]}")

Canceled rentals before deletion: 21310 - Canceled rentals after deletion: 21309


Get the extreme values of ```delay_at_checkout``` = late checkout > 24h

In [12]:
# Get the apparent extreme values above abs(1440 min) delay_at_checkout
mask = np.abs(df_rentals["delay_at_checkout"]) > 1440
df_extreme_delays = df_rentals.loc[mask, :].copy()
print(f"Count of rentals with an absolute 'delay_at_checkout' > 1440 min: {df_extreme_delays.shape[0]}")

# Get the count of ended rentals
df_ended = df_rentals.loc[df_rentals["state"] == "ended"].copy()
print(f"Percentage of rentals beyond this threshold: {100 * df_extreme_delays.shape[0] / df_ended.shape[0]:.2f}%")

# Get the checkouts that are delayed by more than 24h
mask = df_rentals["delay_at_checkout"] > 1440
df_extreme_positive_delays = df_rentals.loc[mask, :].copy()
print("-" * 80)
print(f"Count of checkouts that are delayed by more than 24h: {df_extreme_positive_delays.shape[0]}")
print(f"Percentage those checkouts: {100 * df_extreme_positive_delays.shape[0] / df_ended.shape[0]:.2f}%")

Count of rentals with an absolute 'delay_at_checkout' > 1440 min: 228
Percentage of rentals beyond this threshold: 1.26%
--------------------------------------------------------------------------------
Count of checkouts that are delayed by more than 24h: 188
Percentage those checkouts: 1.04%


In [13]:
# Get the rents within the range of +/- 24h
# = Get the average delays values (within the range [-1440, 1440] min)
mask = np.abs(df_rentals["delay_at_checkout"]) <= 1440
df_average_delays = df_rentals.loc[mask, :].copy()

fig = px.histogram(
    df_average_delays,
    x="delay_at_checkout",
    color="state",
    barmode="overlay",
    nbins=int(np.sqrt(df_average_delays.shape[0])),
    title="Distribution of delay_at_checkout",
    labels={"delay_at_checkout": "delay_at_checkout (minutes)"},
    width=800,
    height=250
    )

# Margin reduction
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()


Analyze missing values

In [14]:
# Count missing values
print(f"Missing values in column {df_rentals['delay_at_checkout'].isna().value_counts()}:")

# Count missing values with state="canceled"
mask = (df_rentals['state'] == 'canceled') & (df_rentals['delay_at_checkout'].isna())
print(f"Nb of lines with state 'canceled': {mask.sum()}")
print(f"Percentage of lines with state 'canceled': {mask.mean() * 100:.2f}%")

# Count missing values with state="ended"
mask = (df_rentals['state'] == 'ended') & (df_rentals['delay_at_checkout'].isna())
print(f"Nb of lines with state 'ended': {mask.sum()}")
print(f"Percentage of lines with state 'ended': {mask.mean() * 100:.2f}%")


Missing values in column delay_at_checkout
False    16345
True      4964
Name: count, dtype: int64:
Nb of lines with state 'canceled': 3264
Percentage of lines with state 'canceled': 15.32%
Nb of lines with state 'ended': 1700
Percentage of lines with state 'ended': 7.98%


Out of the 23% of missing values in ```delay_at_checkout```:  
- 15% of missing "canceled" -> no checkout when cancellation
- 8% of missing "ended".

---
### 🔎 ```delta_with_previous``` analysis

🎯 We want to understand when a delay at checkout might lead to a cancellation.  
Let's try to identify the range of delta values that could lead to cancellations.

In [15]:
# Display the count of canceled among the ended rentals as a function of delta_with_previous
fig = px.histogram(
    df_rentals,
    'delta_with_previous',
    color='state',
    barmode="overlay",
    nbins=int(np.sqrt(df_rentals.shape[0])),
    title="Ended vs Canceled rentals by delta_with_previous",
    labels={"delta_with_previous": "delta in min"},
    width=800,
    height=250,
    )
# Margin reduction
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

- Conclusion: Cancellations occur for all delta intervals.

---
### Analyze cancellations reasons:  
Since we cannot infer causality directly, we need to further analyze cancellations <u>among cars with prior rentals</u> to assess the potential impact of ```delay_at_checkout``` on ```delta_with_previous```.

In [16]:
# Select rentals of cars rented several times a day (previous_ended_rental_id not missing)
df_multi_rentals = df_rentals.loc[df_rentals["previous_ended_rental_id"].notna()].copy()
print(f"{df_multi_rentals.shape[0]} rentals several times a day")
print(f"Percentage of those rentals on the whole dataset: {100 * df_multi_rentals.shape[0] / df_rentals.shape[0]:.2f}%")
print("="*80)

# Are there "canceled" rentals with previous rental in the day ?
df_canceled = df_rentals[df_rentals["state"] == "canceled"].copy()
count_previous_ended_rental_id = df_canceled["previous_ended_rental_id"].notna().sum()
print(f"{count_previous_ended_rental_id}/{df_multi_rentals.shape[0]} Canceled")

# Associate each rental and the delay at checkout of its previous rental
# Create a dictionary for future mapping rental_id -> delay_at_checkout
delay_dict = df_rentals.set_index('rental_id')['delay_at_checkout'].to_dict()
# Add to each rental, the delay_at_checkout of the previous rental
df_multi_rentals['delay_of_previous_checkout'] = df_multi_rentals['previous_ended_rental_id'].map(delay_dict)
# Set the index to rental_id
df_multi_rentals = df_multi_rentals.set_index('rental_id')

# Modify columns order
l_columns = ['previous_ended_rental_id',
             'delay_of_previous_checkout',
             'delta_with_previous',
             'car_id',
             'state',
             'delay_at_checkout',
             'checkin_type'
             ]
df_multi_rentals = df_multi_rentals[l_columns]

# Get canceled rentals because of a previous checkout too late (= delay of previous > scheduled delta with previous)
df_multi_rentals_canceled = df_multi_rentals[df_multi_rentals["state"] == "canceled"].copy()
df_multi_rentals_canceled_for_delay = df_multi_rentals_canceled[df_multi_rentals_canceled["delay_of_previous_checkout"] > df_multi_rentals_canceled["delta_with_previous"]].copy()
print(f"{df_multi_rentals_canceled_for_delay.shape[0]}/{count_previous_ended_rental_id} Canceled rentals because of a late previous checkout")
display(df_multi_rentals_canceled_for_delay)

# Get rentals ended despite a previous checkout too late
print("-"*80)
df_multi_rentals_ended = df_multi_rentals[df_multi_rentals["state"] == "ended"].copy()
df_multi_rentals_ended_despite_delay = df_multi_rentals_ended[df_multi_rentals_ended["delay_of_previous_checkout"] > df_multi_rentals_ended["delta_with_previous"]].copy()
print(f"{df_multi_rentals_ended_despite_delay.shape[0]}/{count_previous_ended_rental_id} Ended rentals despite a late previous checkout")
display(df_multi_rentals_ended_despite_delay)

1841 rentals several times a day
Percentage of those rentals on the whole dataset: 8.64%
229/1841 Canceled
37/229 Canceled rentals because of a late previous checkout


,previous_ended_rental_id,delay_of_previous_checkout,delta_with_previous,car_id,state,delay_at_checkout,checkin_type
rental_id,,,,,,,
539151,548646,201.0,30,282893,canceled,<NA>,mobile
553139,547650,410.0,240,297851,canceled,<NA>,mobile
559037,552392,46.0,0,349171,canceled,<NA>,connect
543745,521852,650.0,150,359045,canceled,<NA>,connect
538224,534999,346.0,180,340014,canceled,<NA>,connect
551372,547240,550.0,210,261576,canceled,<NA>,mobile
568105,564144,210.0,120,351546,canceled,<NA>,connect
546930,551866,153.0,0,316449,canceled,<NA>,mobile
556126,548864,527.0,510,396250,canceled,<NA>,mobile


--------------------------------------------------------------------------------
181/229 Ended rentals despite a late previous checkout


,previous_ended_rental_id,delay_of_previous_checkout,delta_with_previous,car_id,state,delay_at_checkout,checkin_type
rental_id,,,,,,,
540479,539751,3.0,0,374684,ended,12,mobile
541862,540607,1.0,0,382364,ended,125,mobile
559781,540868,26.0,0,408776,ended,44,mobile
574568,572909,13.0,0,301512,ended,110,mobile
535519,533413,4.0,0,353425,ended,-166,connect
...,...,...,...,...,...,...,...
556928,556011,99.0,60,245154,ended,-99,connect
561206,554958,183.0,30,312603,ended,10,connect
561476,550186,21.0,0,410402,ended,11,mobile


In [17]:
# Visualize overlap between delay of previous and scheduled delta
df_multi_rentals["overlap"] = df_multi_rentals["delay_of_previous_checkout"] - df_multi_rentals["delta_with_previous"]

# Delete outliers for display (outliers: |delay| > 2 days)  
df_multi_rentals = df_multi_rentals[np.abs(df_multi_rentals["overlap"]) < 2880].copy()

# Display the count of canceled among the ended rentals as a function of delta_with_previous
fig = px.histogram(
    df_multi_rentals,
    'overlap',
    color='state',
    barmode="overlay",
    #nbins=int(np.sqrt(df_no_outliers.shape[0])),
    title="Ended vs Canceled rentals by overlap",
    labels={"overlap": "overlap (min): positive means late"},
    width=800,
    height=350,
    )
# Margin reduction
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

- The vast majority of checkouts, even delayed are within the scheduled delta. There are some cancellations with compatible checkouts.

In [18]:
# Visualize proportion of multiple rentals per day vs. single rentals per day 
# Create a boolean column: True the car has already been rented the same day
df_rentals["has_previous_rental"] = df_rentals["previous_ended_rental_id"].notna()

# Count the number of rentals with a previous one the same day
df_counts = df_rentals["has_previous_rental"].value_counts().reset_index()
df_counts.columns = ["multiple_rentals_per_day", "count"]

fig = px.pie(
    df_counts,
    names="multiple_rentals_per_day",
    values="count",
    title="Multiple rentals vs. single rental per day",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set2
)
# Margin reduction
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

# Drop added column
df_rentals.drop("has_previous_rental", axis=1, inplace=True)

- Only 1841 rentals (8.64% of rentals) are made several times a day. 

In [19]:
# Visualize proportion of cancelled rentals among multiple rentals per day vs. all rentals
# Create a boolean column: True the car has already been rented the same day and state is canceled
df_rentals["canceled_after_rentals"] = (
    (df_rentals["previous_ended_rental_id"].notna()) &
    (df_rentals["state"] == "canceled")
)

# Count the number of cancelled among the rentals with a previous one the same day
df_counts = df_rentals["canceled_after_rentals"].value_counts().reset_index()
df_counts.columns = ["canceled_after_rentals", "count"]

fig = px.pie(
    df_counts,
    names="canceled_after_rentals",
    values="count",
    title="Canceled rentals among multiple rentals per day vs all rentals",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set3
)
# Margin reduction
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

# Drop added column
df_rentals.drop("canceled_after_rentals", axis=1, inplace=True)

- There are 229 cancellations of rentals for cars already rented in the day. 

In [20]:
# Proportion of canceled rentals due to previous delay
# Count
nb_canceled_due_to_delay = df_multi_rentals_canceled_for_delay.shape[0]
nb_canceled_total = df_multi_rentals_canceled.shape[0]
nb_canceled_other = nb_canceled_total - nb_canceled_due_to_delay

# Create a summary dataframe
df_canceled_summary = pd.DataFrame({
    "reason": ["Due to previous delay", "Other reasons"],
    "count": [nb_canceled_due_to_delay, nb_canceled_other]
})

fig = px.pie(
    df_canceled_summary,
    names="reason",
    values="count",
    title="Proportion of canceled rentals due to previous delay",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set2
)
# Margin reduction
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()


- On these cancellations, there seem to be only 37 cancellations because of a late previous checkout.  
- 181 of delay and delta overlaps still lead to completed rentals. Consumers seem to wait.  Might depend on the rental duration ?

---
### 🔎```checkin_type``` analysis

Analyze the impact of the ```checkin_type``` on the cancellations

In [21]:
# Define function to compute the proportion of cancellations because of a late previous checkout
def get_canceled_summary(df, checkin_type):
    df_filtered = df[df["checkin_type"] == checkin_type]
    df_canceled = df_filtered[df_filtered["state"] == "canceled"]
    df_canceled_for_delay = df_canceled[df_canceled["delay_of_previous_checkout"] > df_canceled["delta_with_previous"]]

    nb_canceled_due_to_delay = df_canceled_for_delay.shape[0]
    nb_canceled_total = df_canceled.shape[0]
    nb_canceled_other = nb_canceled_total - nb_canceled_due_to_delay

    return pd.DataFrame({
        "reason": ["Due to previous delay", "Other reasons"],
        "count": [nb_canceled_due_to_delay, nb_canceled_other],
        "checkin_type": checkin_type
    })


# Create 2 summary dataframes
df_mobile_summary = get_canceled_summary(df_multi_rentals, "mobile")
df_connect_summary = get_canceled_summary(df_multi_rentals, "connect")

# Combine both
df_summary = pd.concat([df_mobile_summary, df_connect_summary])

# 2 pies side by side
fig = px.pie(
    df_summary,
    names="reason",
    values="count",
    facet_col="checkin_type",
    title="Proportion of canceled rentals due to late previous delay by check-in type",
    color_discrete_sequence=px.colors.qualitative.Set2,
    width=1000,
    height=250
)

# Customize labels of each pie
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# Margin reduction
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

- There seems to be a difference of behavior between the ```mobile``` and ```connect``` users. Apparently, the ```mobile``` users cancel rental more often when facing late previous checkout.

---
### Save dataset

In [22]:
# Save df as pickle locally
path_to_data =  f"../data/raw/df_rentals.pkl"   # Save in raw since it's just preprocessed
df_rentals.to_pickle(path_to_data)

---
---
# 🔬 Pricing study
---

In [23]:
# Import data
path_to_data =  f"../data/raw/get_around_pricing_project.csv"
df_cars = pd.read_csv(path_to_data, index_col=0)
df_cars.head()

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183


In [24]:
df_cars.describe(include="all")

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
count,4843,4.843000e+03,4843.00000,4843,4843,4843,4843,4843,4843,4843,4843,4843,4843,4843.000000
unique,28,NaN,NaN,4,10,8,2,2,2,2,2,2,2,NaN
top,Citroën,NaN,NaN,diesel,black,estate,True,True,False,False,False,False,True,NaN
freq,969,NaN,NaN,4641,1633,1606,2662,3839,3865,3881,2613,3674,4514,NaN
mean,NaN,1.409628e+05,128.98823,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.214536
std,NaN,6.019674e+04,38.99336,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.568268
min,NaN,-6.400000e+01,0.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.000000
25%,NaN,1.029135e+05,100.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.000000
50%,NaN,1.410800e+05,120.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,119.000000
75%,NaN,1.751955e+05,135.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136.000000


---
### Missing values

In [25]:
# Get missing values by column
missing_counts = df_cars.isna().sum().reset_index()
missing_counts.columns = ['column', 'missing_count']
display(missing_counts)

,column,missing_count
0,model_key,0
1,mileage,0
2,engine_power,0
3,fuel,0
4,paint_color,0
5,car_type,0
6,private_parking_available,0
7,has_gps,0
8,has_air_conditioning,0
9,automatic_car,0


No missing value

---
### Column visualization

In [26]:
# Get column types
display(df_cars.dtypes)

model_key                    object
mileage                       int64
engine_power                  int64
fuel                         object
paint_color                  object
car_type                     object
private_parking_available      bool
has_gps                        bool
has_air_conditioning           bool
automatic_car                  bool
has_getaround_connect          bool
has_speed_regulator            bool
winter_tires                   bool
rental_price_per_day          int64
dtype: object

In [27]:
# Display proportion of each category in every categorical column
# CATEGORICAL COLUMNS
# Get list of categorical columns
l_categorical_cols = df_cars.select_dtypes(include=["object", "category", "bool"]).columns

# Number of lines needed
rows = (len(l_categorical_cols) + 1) // 2

fig = make_subplots(
    rows=rows,
    cols=2,
    subplot_titles=l_categorical_cols,
    specs=[[{"type": "domain"}, {"type": "domain"}] for _ in range(rows)]
)

# Display proportion of each category in every categorical column
for i, col in enumerate(l_categorical_cols):
    counts = df_cars[col].value_counts().reset_index()
    counts.columns = [col, "count"]
    pie = go.Pie(labels=counts[col], values=counts["count"], name=col, textinfo="percent+label")
    fig.add_trace(pie, row=i // 2 + 1, col=i % 2 + 1)

fig.update_layout(
    title_text="Categorical columns distribution",
    height=440 * rows,
    width=800,
    showlegend=False,
    margin=dict(l=10, r=10, t=10, b=10)  # left, right, top, bottom
)
fig.show()

# NUMERIC COLUMNS
l_numeric_cols = df_cars.select_dtypes(include=["int64", "float64"]).columns
rows = (len(l_numeric_cols) + 1) // 2

fig = make_subplots(
    rows=rows,
    cols=2,
    subplot_titles=l_numeric_cols
)

for i, col in enumerate(l_numeric_cols):
    hist = go.Histogram(x=df_cars[col], name=col)
    fig.add_trace(hist, row=i // 2 + 1, col=i % 2 + 1)

fig.update_layout(
    title_text="Numeric columns distribution",
    height=250 * rows,
    width=800,
    showlegend=False,
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
)

fig.show()

- ```model_key```: Citroen + Renault + Peugeot + BMW + Audi represent more than 80% of cars.
- ```diesel```: Diesel (95.8%) + Petrol represent almost 100% of the market.
- ```paint_color```: Neutral colors (Black + Grey + White + Brown + Silver) represent 83% of the cars. Among fancy colors, blue is the most represented.
- ```car_type```: Estate, Sedan, SUV, Hatchback represent almost 93.5% of the cars (almost evenly distributed).
- ```private_parking_available```: almost half of the cars have a private parking.
- 20% of cars have gps, air conditioning, speed regulator, or are automatic.
- half of the cars have ```has_getaround_connect```
- More than 93% of the cars have winter tyres.

---
### Numeric columns analysis

##### Target: ```Distribution of rental_price_per_day```

The distribution looks like a typical bell shape, with some outliers.  
Apply the 3 sigmas rule to discard them.
- outliers analysis

In [28]:

# Outliers identification
# Apply 3 sigmas rule
mean_price = np.mean(df_cars['rental_price_per_day'])
three_sigmas = 3*np.std(df_cars['rental_price_per_day'])

low_limit = mean_price - three_sigmas
high_limit = mean_price + three_sigmas
print(f"Prices within 3 sigmas interval in [{low_limit:.2f}; {high_limit:.2f}]")

fig = px.histogram(
    df_cars,
    x="rental_price_per_day",
    title="Rental price per day",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set2
    )

fig.add_vline(x=low_limit, line_dash = 'dash', line_color = 'steelblue')
fig.add_vline(x=high_limit, line_dash = 'dash', line_color = 'steelblue')
fig.add_vline(x=mean_price,  line_color = 'steelblue')

# Margin reduction
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

Prices within 3 sigmas interval in [20.52; 221.91]


In [29]:

# Analyze distribution of values in below limit prices outliers
df_cars_below_price_limit = df_cars[df_cars['rental_price_per_day'] < low_limit].copy()
df_cars_below_price_limit

# CATEGORICAL COLUMNS
# Get list of categorical columns
l_categorical_cols = df_cars_below_price_limit.select_dtypes(include=["object", "category"]).columns

# Number of lines needed
rows = (len(l_categorical_cols) + 1) // 2

fig = make_subplots(
    rows=rows,
    cols=2,
    subplot_titles=l_categorical_cols,
    specs=[[{"type": "domain"}, {"type": "domain"}] for _ in range(rows)]
)

# Display proportion of each category in every categorical column
for i, col in enumerate(l_categorical_cols):
    counts = df_cars_below_price_limit[col].value_counts().reset_index()
    counts.columns = [col, "count"]
    pie = go.Pie(labels=counts[col], values=counts["count"], name=col, textinfo="percent+label")
    fig.add_trace(pie, row=i // 2 + 1, col=i % 2 + 1)

fig.update_layout(
    title_text="Categorical columns distribution for prices outliers below 3 sigmas limit",
    height=250 * rows,
    width=800,
    showlegend=False,
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
)
fig.show()

# NUMERIC COLUMNS
l_numeric_cols = df_cars_below_price_limit\
    .drop("rental_price_per_day", axis=1)\
    .select_dtypes(include=["int64", "float64"]).columns
rows = (len(l_numeric_cols) + 1) // 2

fig = make_subplots(
    rows=rows,
    cols=2,
    subplot_titles=l_numeric_cols
)

for i, col in enumerate(l_numeric_cols):
    hist = go.Histogram(x=df_cars_below_price_limit[col], name=col)
    fig.add_trace(hist, row=i // 2 + 1, col=i % 2 + 1)

fig.update_layout(
    title_text="Numeric columns distribution for prices outliers below 3 sigmas limit",
    height=250 * rows,
    width=800,
    showlegend=False,
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
)

fig.show()

In [30]:
# Analyze distribution of values in above limit prices outliers
df_cars_above_price_limit = df_cars[df_cars['rental_price_per_day'] > high_limit].copy()
df_cars_above_price_limit

# CATEGORICAL COLUMNS
# Get list of categorical columns
l_categorical_cols = df_cars_above_price_limit.select_dtypes(include=["object", "category"]).columns

# Number of lines needed
rows = (len(l_categorical_cols) + 1) // 2

fig = make_subplots(
    rows=rows,
    cols=2,
    subplot_titles=l_categorical_cols,
    specs=[[{"type": "domain"}, {"type": "domain"}] for _ in range(rows)]
)

# Display proportion of each category in every categorical column
for i, col in enumerate(l_categorical_cols):
    counts = df_cars_above_price_limit[col].value_counts().reset_index()
    counts.columns = [col, "count"]
    pie = go.Pie(labels=counts[col], values=counts["count"], name=col, textinfo="percent+label")
    fig.add_trace(pie, row=i // 2 + 1, col=i % 2 + 1)

fig.update_layout(
    title_text="Categorical columns distribution for prices outliers above 3 sigmas limit",
    height=250 * rows,
    width=800,
    showlegend=False,
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
)
fig.show()

# NUMERIC COLUMNS
l_numeric_cols = df_cars_above_price_limit\
    .drop("rental_price_per_day", axis=1)\
    .select_dtypes(include=["int64", "float64"]).columns
rows = (len(l_numeric_cols) + 1) // 2

fig = make_subplots(
    rows=rows,
    cols=2,
    subplot_titles=l_numeric_cols
)

for i, col in enumerate(l_numeric_cols):
    hist = go.Histogram(x=df_cars_above_price_limit[col], name=col)
    fig.add_trace(hist, row=i // 2 + 1, col=i % 2 + 1)

fig.update_layout(
    title_text="Numeric columns distribution for prices outliers above 3 sigmas limit",
    height=250 * rows,
    width=800,
    showlegend=False,
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
)
fig.show()

- Outliers below limit are of mainstream models (Renault, Citroen, Peugeot, Audi, BMW), with high mileage, of Sedan or Estate type.
- Outliers above limit are of more luxurious models, with low mileage, high engine power, SUV or Sedan type.

-> Delete outliers

In [31]:

# Keep prices values between mean +/- 3 sigmas
df_cars = df_cars[(df_cars["rental_price_per_day"] >= low_limit) & (df_cars["rental_price_per_day"] <= high_limit)].copy()

fig = px.histogram(
    df_cars,
    x="rental_price_per_day",
    title="Rental price per day",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set2
    )

fig.add_vline(x=low_limit, line_dash = 'dash', line_color = 'steelblue')
fig.add_vline(x=high_limit, line_dash = 'dash', line_color = 'steelblue')
fig.add_vline(x=mean_price,  line_color = 'steelblue')
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )

fig.show()

---
##### ```mileage``` analysis

The distribution shows extreme values.  
- outliers analysis

In [32]:

# Display distribution
fig = px.histogram(
    df_cars,
    x="mileage",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

fig = px.box(
    df_cars,
    x="mileage",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

In [33]:
# Get mileage > 400000
df_cars_high_mileage = df_cars[df_cars["mileage"] > 400000].copy()
display(df_cars_high_mileage)

# Get extreme mileage
df_cars_high_mileage = df_cars[df_cars["mileage"] <= 1000].copy()
display(df_cars_high_mileage)

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
557,Renault,484615,120,diesel,blue,estate,True,True,False,False,False,False,True,91
1573,Citroën,400654,110,diesel,black,estate,False,False,True,False,False,False,True,42
2350,Peugeot,477571,85,diesel,grey,hatchback,False,True,False,False,False,True,False,35
3198,Citroën,405816,100,diesel,blue,sedan,False,False,False,False,False,False,True,22
3732,Citroën,1000376,90,diesel,black,subcompact,True,False,False,False,False,False,True,37


,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
2409,Opel,476,120,diesel,blue,hatchback,True,True,False,False,False,True,True,174
3935,Mitsubishi,706,155,diesel,black,suv,True,True,False,True,True,True,True,204


1 million km mileage seems completely out of range.
-> delete the line

In [34]:
# Delete outlier in mileage
cars_count = df_cars.shape[0]
df_cars = df_cars[df_cars["mileage"] < 1000000].copy()
print(f"Count of deleted outliers in 'mileage' column: {cars_count - df_cars.shape[0]}")

# Display distribution
fig = px.histogram(
    df_cars,
    x="mileage",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

Count of deleted outliers in 'mileage' column: 1


---
##### ```engine_power``` analysis

The distribution shows extreme values.  
- outliers analysis

In [35]:
# Display engine_power distribution
fig = px.histogram(
    df_cars,
    x="engine_power",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set1
    )
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

fig = px.box(
    df_cars,
    x="engine_power",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set1
    )
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

In [36]:
# Look for outliers
# Get engine_power <= 25
mask = (df_cars["engine_power"] <= 25) | (df_cars["engine_power"] > 320)
df_cars_engine_power = df_cars[mask].copy()
display(df_cars_engine_power)


,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
1796,Porsche,152328,25,hybrid_petrol,black,hatchback,False,True,False,False,False,False,True,142
1925,Porsche,152470,25,hybrid_petrol,black,hatchback,False,True,False,False,False,False,True,124
3601,Mini,150187,412,petrol,white,sedan,True,True,True,False,True,True,True,204
3765,Nissan,81770,0,diesel,white,suv,False,False,False,False,False,False,False,108


There is no car (at least useful) with an engine_power of 0.  
There are no Porsche with 25 of engine power (KW ou HP), neither is there a Mini with a 412 engine power.  
-> remove the lines

In [37]:
# Delete outliers in engine_power
cars_count = df_cars.shape[0]
df_cars = df_cars[~mask].copy()
print(f"Count of deleted outliers in 'engine_power' column: {cars_count - df_cars.shape[0]}")

fig = px.histogram(
    df_cars,
    x="engine_power",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set1
    )
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

Count of deleted outliers in 'engine_power' column: 4


---
### Analyze individual relation of each categorical column with the target

In [38]:
df_cars

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183
5,Citroën,152352,225,petrol,black,convertible,True,True,False,False,True,True,True,131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4838,Toyota,39743,110,diesel,black,van,False,True,False,False,False,False,True,121
4839,Toyota,49832,100,diesel,grey,van,False,True,False,False,False,False,True,132
4840,Toyota,19633,110,diesel,grey,van,False,True,False,False,False,False,True,130
4841,Toyota,27920,110,diesel,brown,van,True,True,False,False,False,False,True,151


In [39]:
# Get list of categorical columns
# CATEGORICAL COLUMNS
l_categorical_cols = df_cars.select_dtypes(include=["object", "category", "bool"]).columns

# Display the count of rental_price_per_day for each categorical value in every categorical column
for _, col in enumerate(l_categorical_cols):
    fig = px.histogram(
        df_cars,
        x="rental_price_per_day",
        color=col,  # get the current categorical column
        nbins=40,
        barmode="overlay",
        title=f"Distribution of rental_price_per_day by {col}",
        height=450,
        width=1000
    )
    fig.show()

- ```rental_price_per_day``` distributions by all categorical values show different shapes depending on the values of every categorical column.

---
### Save dataset

In [40]:
# Save df as pickle locally
path_to_data =  f"../data/raw/df_cars.pkl"   # Save in raw since it's just preprocessed
df_cars.to_pickle(path_to_data)

---
---
# Q&A
---

##### Import data

In [41]:
# Import rentals and cars dataframes
path_to_data =  f"../data/raw/df_rentals.pkl"
df_rentals = pd.read_pickle(path_to_data)
print(f"Dimensions of df_rentals: {df_rentals.shape}")
display(df_rentals.head(5))

path_to_data =  f"../data/raw/df_cars.pkl"
df_cars = pd.read_pickle(path_to_data)
print(f"Dimensions of df_cars: {df_cars.shape}")
display(df_cars.head(5))

Dimensions of df_rentals: (21309, 7)


,rental_id,car_id,checkin_type,state,delay_at_checkout,previous_ended_rental_id,delta_with_previous
0,505000,363965,mobile,canceled,<NA>,<NA>,<NA>
1,507750,269550,mobile,ended,-81,<NA>,<NA>
2,508131,359049,connect,ended,70,<NA>,<NA>
3,508865,299063,connect,canceled,<NA>,<NA>,<NA>
4,511440,313932,mobile,ended,<NA>,<NA>,<NA>


Dimensions of df_cars: (4784, 14)


,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183
5,Citroën,152352,225,petrol,black,convertible,True,True,False,False,True,True,True,131


### 1. How many rentals would be affected by the feature depending on the threshold and scope we choose?

In [42]:
df_rentals.columns

Index(['rental_id', 'car_id', 'checkin_type', 'state', 'delay_at_checkout',
       'previous_ended_rental_id', 'delta_with_previous'],
      dtype='object')

#### scope: connect cars

Combien de locations ont un delta_with_previous < seuil ?

In [43]:
# Select connect checkin
mask = df_rentals["checkin_type"] == 'connect'
df_rentals_connect = df_rentals[mask].copy()
locations_connect_count = df_rentals_connect.shape[0]
print(f"Locations of connect type: {locations_connect_count}")

# Define threshold
THRESHOLD = 60

# Use of np.where
condition = (df_rentals_connect['delta_with_previous'] <= THRESHOLD) & (df_rentals_connect['delta_with_previous'].notna())
df_rentals_connect['threshold'] = np.where(condition, 'below', 'above')

# Count values in threshold
df_rentals_connect['threshold'].value_counts()
count_below = df_rentals_connect[df_rentals_connect['threshold'] == 'below'].shape[0]

fig = px.pie(
    df_rentals_connect['threshold'].value_counts(),
    values='count',
    names=df_rentals_connect['threshold'].value_counts().index, 
    color=df_rentals_connect['threshold'].value_counts().index,
    title=f"Connect rentals by threshold = {THRESHOLD}: {count_below} set aside on {locations_connect_count} locations",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set2,
    )
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

Locations of connect type: 4307


#### scope: all cars

In [44]:
# Use of np.where
condition = (df_rentals['delta_with_previous'] <= THRESHOLD) & (df_rentals['delta_with_previous'].notna())
df_rentals['threshold'] = np.where(condition, 'below', 'above')
locations_count = df_rentals.shape[0]
print(f"All types of locations: {locations_count}")

# Count values in threshold
df_rentals['threshold'].value_counts()
count_below = df_rentals[df_rentals['threshold'] == 'below'].shape[0]

fig = px.pie(
    df_rentals['threshold'].value_counts(),
    values='count',
    names=df_rentals['threshold'].value_counts().index, 
    color=df_rentals['threshold'].value_counts().index,
    title=f"All rentals by threshold = {THRESHOLD}: {count_below} set aside on {locations_count} locations",
    width=600,
    height=250,
    color_discrete_sequence=px.colors.qualitative.Set2,
    )
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

df_rentals.drop("threshold", axis=1, inplace=True)

All types of locations: 21309


Conclusion:
- Appliqué sur les locations via connect, un délai tampon de 60 min entre 2 locations, fait perdre 260 locations, soit 6% des 4307 locations.
- Sur l'ensemble des locations, le même délai fait perdre un peu plus du double, soit 584 locations, correspondant à 2,7% des 21309 locations.

---

### 2. How often are drivers late for the next check-in? How does it impact the next driver?

Sur l'ensemble des locations avec une heure de retour renseignée (delay_at_checkout non nulle), on observe le ratio retard/avance.

In [45]:
# Get all the rentals, even the cars rented only once a day
# Get the rentals for which we know the delay at checkout
df_rentals_noNA = df_rentals.loc[df_rentals["delay_at_checkout"].notna()].copy()
print(f"df_rentals_noNA.shape[0]: {df_rentals_noNA.shape[0]}")

# Create column delay_label with value "advance/late"
df_rentals_noNA["delay_label_with_next"] = df_rentals_noNA["delay_at_checkout"].apply(lambda x: "advance" if x<=0 else "late")

# Display the ratio advance/late
fig = px.pie(
    df_rentals_noNA,
    names='delay_label_with_next',
    color=df_rentals_noNA['delay_label_with_next'],
    title='Ratio advance/late among drivers for all rentals (including uniques)',
    width=600,
    height=250,
    color_discrete_map={"advance": "lightgreen", "late": "steelblue"},
    )
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

df_rentals_noNA.shape[0]: 16345


Pour étudier l'impact du retard sur la location suivante, il faut sélectionner les voitures qui sont utilisées plusieurs fois dans la même journée.

In [46]:
# Get ONLY the rentals of cars rented several times a day
df_multi_rentals_noNA = df_rentals_noNA.loc[df_rentals_noNA["previous_ended_rental_id"].notna()].copy()

# Display the ratio advance/late
fig = px.pie(
    df_multi_rentals_noNA['delay_label_with_next'].value_counts(),
    values='count',
    names=df_multi_rentals_noNA['delay_label_with_next'].value_counts().index, 
    color=df_multi_rentals_noNA['delay_label_with_next'].value_counts().index,
    title='Ratio advance/late among drivers for cars with several rentals per day',
    width=600,
    height=250,
    color_discrete_map={"advance": "lightgreen", "late": "steelblue"},
    )
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

Drivers are late more than half of the time.

In [47]:
df_multi_rentals_noNA

,rental_id,car_id,checkin_type,state,delay_at_checkout,previous_ended_rental_id,delta_with_previous,delay_label_with_next
6,511639,370585,connect,ended,-15,563782,570,advance
19,519491,312389,mobile,ended,58,545639,420,late
40,528808,181625,connect,ended,-76,557404,330,advance
64,533670,320824,connect,ended,-6,556563,630,advance
74,534827,404169,mobile,ended,-7,531158,90,advance
...,...,...,...,...,...,...,...,...
21249,571823,353425,connect,ended,-276,569556,240,advance
21253,573274,298117,connect,ended,-7,571227,210,advance
21266,567741,294059,mobile,ended,111,567708,120,late
21275,568523,297973,mobile,ended,12,567121,240,late


Pour étudier l'impact sur la location suivante d'un retour hors délai prévu, il faut référencer le delay_at_checkout de la précédente location dans une colonne pour la location courante.  
Remarque: on prend toutes les locations, même celles qui sont uniques dans la journée ou celles pour lesquelles l'heure de retour n'est pas mentionnée (annulation), pour observer l'effet du retard (annulation) sur la location courante.

In [48]:
# Merge current rental info and the previous rental of the same car if possible
df_rentals_merged = pd.merge(
    df_rentals,
    df_rentals[["rental_id", "delay_at_checkout"]],
    how='left',
    left_on='previous_ended_rental_id',
    right_on='rental_id',
    suffixes=('_current', '_previous')
    )
print(df_rentals_merged.columns)

# Rename columns
l_cols = ['rental_id', 'car_id', 'checkin_type', 'state',
       'delay_at_checkout', 'previous_ended_rental_id',
       'delta_with_previous', 'rental_id_previous',
       'previous_delay']
df_rentals_merged.columns = l_cols
# Drop redundant joint key column
df_rentals_merged.drop("rental_id_previous", axis=1, inplace=True) 

# Identify ovelaps conflicts case in a new column
df_rentals_merged["checkin_conflict"] = (
    df_rentals_merged["previous_delay"].notna() &
    df_rentals_merged["delta_with_previous"].notna() &
    (df_rentals_merged["previous_delay"] > df_rentals_merged["delta_with_previous"])
    )
df_rentals_merged 

Index(['rental_id_current', 'car_id', 'checkin_type', 'state',
       'delay_at_checkout_current', 'previous_ended_rental_id',
       'delta_with_previous', 'rental_id_previous',
       'delay_at_checkout_previous'],
      dtype='object')


,rental_id,car_id,checkin_type,state,delay_at_checkout,previous_ended_rental_id,delta_with_previous,previous_delay,checkin_conflict
0,505000,363965,mobile,canceled,<NA>,<NA>,<NA>,<NA>,False
1,507750,269550,mobile,ended,-81,<NA>,<NA>,<NA>,False
2,508131,359049,connect,ended,70,<NA>,<NA>,<NA>,False
3,508865,299063,connect,canceled,<NA>,<NA>,<NA>,<NA>,False
4,511440,313932,mobile,ended,<NA>,<NA>,<NA>,<NA>,False
...,...,...,...,...,...,...,...,...,...
21304,573446,380069,mobile,ended,<NA>,573429,300,<NA>,False
21305,573790,341965,mobile,ended,-337,<NA>,<NA>,<NA>,False
21306,573791,364890,mobile,ended,144,<NA>,<NA>,<NA>,False
21307,574852,362531,connect,ended,-76,<NA>,<NA>,<NA>,False


On peut étudier les taux d'annulation de location en fonction des conflits de checkin

In [49]:
# Create df of cancellations rates
# Filter only rentals with a previous rental ie previous_delay notna
df_conflict_cancel_rates = df_rentals_merged[df_rentals_merged["previous_delay"].notna()]\
    .groupby("checkin_conflict")["state"]\
    .value_counts(normalize=True)\
    .rename("ratio")\
    .reset_index()
print(len(df_rentals_merged))

# Visualize
fig = px.bar(
    df_conflict_cancel_rates,
    x="checkin_conflict",
    y="ratio",
    color="state",
    barmode="stack",
    text_auto=".1%",
    title="Effect on cancellations rates of checkin conflicts ",
    width=600,
    height=300,
    color_discrete_map={"ended": "lightgreen", "canceled": "steelblue"},
)
fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10),
    legend_title_text="Rental state"
)
fig.show()

21309


Il y a 6 points d'annulations supplémentaires en cas de dépassement du delta prévu pour le retour de location.  
Est-ce que ce taux est différent selon le mode de location (checkin_type) ?

In [50]:
# Get cancellation rates by checkin_conflict AND checkin_type
df_conflict_cancel_rates_by_checkin_type = (
    df_rentals_merged[df_rentals_merged["previous_delay"].notna()]
    .groupby(["checkin_type", "checkin_conflict"])["state"]
    .value_counts(normalize=True)
    .rename("ratio")
    .reset_index()
)
df_conflict_cancel_rates_by_checkin_type

fig = px.bar(
    df_conflict_cancel_rates_by_checkin_type,
    x="checkin_conflict",
    y="ratio",
    color="state",
    barmode="stack",
    facet_col="checkin_type",  # graph per checkin_type
    text_auto=".1%",
    title="Effect on cancellations rates of checkin conflicts depending on checkin_type",
    width=900,
    height=400,
    color_discrete_map={"ended": "lightgreen", "canceled": "steelblue"},
)

fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10),
    legend_title_text="Rental state"
)
fig.show()

Les locations par connect débouchent sur des annulations plus nombreuses en cas de dépassement de délai que les locations par mobile.

---

### 3. How many problematic cases will it solve depending on the chosen threshold and scope?

Un checkout est problématique si la location suivante est impactée par le retard de checkout.

In [51]:
# Filter by threshold
l_thresholds = [0, 60, 120, 180]
l_results = []

for th in l_thresholds:
    df_th = df_rentals_merged[df_rentals_merged["delta_with_previous"] >= th].copy()
    total_conflicts = df_th["checkin_conflict"].sum()
    l_results.append({
        "threshold": th,
        "conflicts": total_conflicts
    })

df_results = pd.DataFrame(l_results)
print(df_results)

fig = px.bar(
    df_results,
    x="threshold",
    y="conflicts",
    text="conflicts",
    title="Problematic cases by threshold of delta_with_previous",
    labels={"threshold": "Threshold (min)", "conflicts": "problematic cases"},
    width=600,
    height=380
)
fig.update_traces(textposition="outside")
fig.update_layout(margin=dict(l=10, r=10, t=40, b=10))
fig.show()

   threshold  conflicts
0          0        218
1         60         72
2        120         38
3        180         22


In [52]:
# Filter by threshold
l_thresholds = [0, 60, 120, 180]
l_results = []

for th in l_thresholds:
    for checkin in ["connect", "mobile"]:
        df_th = df_rentals_merged[
            (df_rentals_merged["delta_with_previous"] >= th) &
            (df_rentals_merged["checkin_type"] == checkin)
        ].copy()
        total_conflicts = df_th["checkin_conflict"].sum()
        l_results.append({
            "threshold": th,
            "checkin_type": checkin,
            "conflicts": total_conflicts
        })


df_results = pd.DataFrame(l_results)
print(df_results)

# Barplot côte à côte
fig = px.bar(
    df_results,
    x="threshold",
    y="conflicts",
    color="checkin_type",
    barmode="group",  # side by side
    text="conflicts",
    title="Problematic cases by threshold of delta_with_previous and checkin_type",
    labels={"threshold": "Threshold (min)", "conflicts": "Problematic cases", "checkin_type": "Check-in type"},
    width=900,
    height=400
)

fig.update_traces(textposition="outside")
fig.update_layout(margin=dict(l=10, r=10, t=50, b=10))
fig.show()

   threshold checkin_type  conflicts
0          0      connect         69
1          0       mobile        149
2         60      connect         21
3         60       mobile         51
4        120      connect         10
5        120       mobile         28
6        180      connect          5
7        180       mobile         17


Le nombre de cas problématiques suit la même tendance entre connect et mobile.  
Un seuil de 60min résoud les 2/3 de cas problématiques.

---

### 4. Which share of our owner’s revenue would potentially be affected by the feature?

Il y a 260 locations de type connect qui ont un delta_with_previous < Seuil.  
Sur ces 260, il y a 69 cas problématiques (cf. ci-dessus).  
Si on applique ce seuil aux locations connect on perd donc 260 locations (cf. Q1.), soit 6% du nombre des <u>locations connect</u> (260/4307).  

Pour estimer la perte de revenu, il faut analyser le dataset cars :  
en filtrant les voitures qui ont getaround_connect, on peut visualiser et analyser les prix de location de ces voitures

In [53]:
df_cars

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183
5,Citroën,152352,225,petrol,black,convertible,True,True,False,False,True,True,True,131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4838,Toyota,39743,110,diesel,black,van,False,True,False,False,False,False,True,121
4839,Toyota,49832,100,diesel,grey,van,False,True,False,False,False,False,True,132
4840,Toyota,19633,110,diesel,grey,van,False,True,False,False,False,False,True,130
4841,Toyota,27920,110,diesel,brown,van,True,True,False,False,False,False,True,151


In [54]:
# Select in dataset cars, cars with the connect option
df_cars_connect = df_cars[df_cars["has_getaround_connect"] == True]
total_connect_cars = len(df_cars_connect)
print(f"Number of cars with connect option: {total_connect_cars}")

# Display
fig = px.histogram(
    df_cars_connect,
    'rental_price_per_day',
    color='has_getaround_connect',
    title="Price distribution of cars with connect option",
    width=600,
    height=300
    )

# Compute business values
# Get mean price
mean_price = df_cars_connect['rental_price_per_day'].mean()
print(f"mean_price: {mean_price}")

fig.add_vline(
    x=mean_price,
    line_dash = 'dash',
    line_color = 'blue'
    )
fig.update_layout(margin=dict(l=10, r=10, t=50, b=10))
fig.show()

Number of cars with connect option: 2221
mean_price: 132.5024763619991


Pour estimer l'impact de l'application du seuil :
On pose les hypothèses suivantes :
- la moyenne des prix des voitures qui ont l'option connect est de 132$ (Q3)  
- le CA est le produit du nombre de locations par la moyenne des prix  
- les locations effectuées et filtrées ont des durées équivalentes, donc le CA est amputé d'un montant proportionnel au nombre de locations filtrées  

On peut donc calculer la perte de CA à cause du filtrage au-delà du seuil.  
Un seuil de 60min minimum entre une location et la relocation suivante concerne 260 locations sur 4307, soit 6% du volume de ces locations de type connect.  
Or sur ces 260 locations, 69 auraient posé un problème d'overlap.  
D'après Q2, il y a 27.5 - 14.1 = 13.4pts d'annulation en plus, en cas de conflit. Soit un % de 13.4/(100-14.1) = 15,6%  
Cela signifie que sur les 69 conflictuels en-dessous du seuil, 15,6% des 69 conflits auraient été annulés, soient 11 cas sur les 69.  
Cela signifie que pour éviter 11 annulations pour conflits de retour, on a annulé 260 locations possibles.  
On a donc réellement perdu 260-11 = 249 locations en appliquant ce seuil, soit une perte de CA de 32993$  
On sélectionne les locations filtrées, c'est-à-dire les 260. Il faudrait écarter 11 locations du calcul parce qu'elles auraient été annulées pour cause de conflit mais on les conserve. En revanche, on regroupe par car_id pour évaluer l'impact du seuil sur des multi locations dans la même journée.  

In [55]:
# Percentage of filtered rentals (below threshold)
# Select connect checkin
df_rentals_connect = df_rentals[df_rentals["checkin_type"] == 'connect'].copy()
locations_connect_count = len(df_rentals_connect)
print(f"Locations of connect type: {locations_connect_count}")

# Define threshold
THRESHOLD = 60

# Use of np.where
condition = (df_rentals_connect['delta_with_previous'] <= THRESHOLD) & (df_rentals_connect['delta_with_previous'].notna())
df_rentals_connect['threshold'] = np.where(condition, 'below', 'above')

# Count values in threshold
df_rentals_connect['threshold'].value_counts()
count_below = df_rentals_connect[df_rentals_connect['threshold'] == 'below'].shape[0]
print(f"Filtered values below threshold: {count_below}")

# Get percentage of filtered rentals
percentage_rentals_below_threshold = 100 * count_below / locations_connect_count
print(f"Percentage of rentals below threshold: {percentage_rentals_below_threshold:.2f}%")

# Get conflicts of connect rentals below threshold
conflicts_rentals = l_results[0]["conflicts"]
print(f"Conflicts of connect rentals below threshold: {conflicts_rentals}")

# Get the difference between cancellations between conflict and no conflict for connect
connect = df_conflict_cancel_rates_by_checkin_type["checkin_type"] == "connect"
canceled = df_conflict_cancel_rates_by_checkin_type["state"] == "canceled"
conflict = df_conflict_cancel_rates_by_checkin_type["checkin_conflict"] == True
diff_canceled_rentals_conflict_below_thhreshold = (
        df_conflict_cancel_rates_by_checkin_type.loc[connect & canceled & conflict, "ratio"].values[0] 
        - df_conflict_cancel_rates_by_checkin_type.loc[connect & canceled & ~conflict, "ratio"].values[0]
        )*100
print(f"Difference between canceled rentals with and without conflict below threshold: {diff_canceled_rentals_conflict_below_thhreshold:.2f} points")

# Get the percentage of cancellations because of conflicts
percentage_canceled_rentals_conflict_below_thhreshold = (
        diff_canceled_rentals_conflict_below_thhreshold 
        / (100 - 100 * df_conflict_cancel_rates_by_checkin_type.loc[connect & canceled & ~conflict, "ratio"].values[0])
        )*100
print(f"Percentage of canceled rentals with conflict below threshold: {percentage_canceled_rentals_conflict_below_thhreshold:.2f}%")

# Canceled conflicts among the conflicts below threshold
canceled_conflicts_rentals = int(percentage_canceled_rentals_conflict_below_thhreshold / 100 * conflicts_rentals) + 1
print(f"Canceled conflicts among the conflicts below threshold: {canceled_conflicts_rentals}")

# Lost rentals apart from those which would have been canceled
lost_rentals = count_below - canceled_conflicts_rentals
print(f"Lost rentals apart from those which would have been canceled: {lost_rentals}")

# Business loss
business_loss = int(lost_rentals * mean_price)
print(f"Total business loss: {business_loss}")

# Group by car_id the filtered connect rentals below threshold
df_filtered = df_rentals_connect[df_rentals_connect['threshold'] == 'below'].copy()

df_rentals_lost =(df_filtered
        .groupby("car_id")
        .size()
        .reset_index(name="count")
        ).copy()
df_rentals_lost["loss"] = np.round(df_rentals_lost["count"] * mean_price, 2)
df_rentals_lost

fig = px.bar(
        df_rentals_lost,
        x="loss",
        y="count",
        title="Threshold impact on business loss for connect cars with and w/o conflicts",
        width=600,
        height=350
        )
fig.update_layout(margin=dict(l=10, r=10, t=50, b=10))
fig.show()

Locations of connect type: 4307
Filtered values below threshold: 260
Percentage of rentals below threshold: 6.04%
Conflicts of connect rentals below threshold: 69
Difference between canceled rentals with and without conflict below threshold: 13.41 points
Percentage of canceled rentals with conflict below threshold: 15.61%
Canceled conflicts among the conflicts below threshold: 11
Lost rentals apart from those which would have been canceled: 249
Total business loss: 32993


Ce résultat suggère que la mesure de sécurité est efficace pour réduire les conflits, mais trop coûteuse en termes de perte de volume.
Des stratégies plus fines (seuils variables, ciblage par véhicule ou période) devraient être envisagées pour limiter cet impact.